# Scheduled Sampling for Autoregressive Graph Generation

This notebook documents the implementation and rationale of scheduled sampling in our BiGG-based graph generator. Scheduled sampling directly addresses the **train-test discrepancy** (exposure bias) by gradually replacing ground-truth inputs with the model's own predictions during training, bridging the gap between teacher-forced training and autoregressive generation.

**Overview:**
1. Why teacher forcing causes hidden state drift at generation time
2. Scheduled sampling: swapping ground truth with model predictions
3. Linear annealing schedule
4. Implementation details: detaching gradients
5. Combining with hidden state noise

## 1. Teacher Forcing and the Train-Test Gap

In BiGG, after predicting node features at step $t$, the model embeds those features and feeds the embedding into the LSTM to produce the hidden state for step $t+1$. During training with teacher forcing, the **ground-truth** features are embedded instead of the model's predictions:

```
Training:   h_t → predict → embed(ground_truth) → LSTM → h_{t+1}
Generation: h_t → predict → embed(prediction)   → LSTM → h_{t+1}
```

The LSTM never learns to recover from its own prediction errors, because during training it always receives perfect inputs. At generation time, prediction errors compound through the feedback loop, causing the hidden state to drift into unfamiliar regions.

This was directly observed in our Reddit experiments: the continuous feature loss reached near-zero (the decoder perfectly mapped clean hidden states to features), but generated graphs had degraded structural and feature quality because the hidden states drifted during generation.

## 2. Scheduled Sampling

Scheduled sampling (Bengio et al., 2015) addresses this by stochastically replacing the ground-truth input with the model's own prediction during training. At each step, with probability $p$ (the **sampling probability**), the model uses its own predicted features for the state update instead of the ground truth.

$$\text{state\_update} = \begin{cases} \text{embed}(\hat{y}_t) & \text{with probability } p \\ \text{embed}(y_t) & \text{with probability } 1-p \end{cases}$$

where $y_t$ is the ground-truth and $\hat{y}_t$ is the model's prediction.

The loss is always computed against the ground truth — scheduled sampling only affects the **state update path**, not the loss. This means the model still receives correct supervision signals while learning to be robust to its own imperfect inputs.

**Key insight:** The model learns to produce hidden states that are useful for prediction even when the input to the LSTM was imperfect. This is exactly the condition it faces during generation.

## 3. Linear Annealing Schedule

The sampling probability $p$ is not fixed — it is annealed from 0 to a maximum value `ss_max_prob` over the course of training. Early in training, the model needs clean inputs to learn basic patterns. As training progresses, the swap probability increases, gradually exposing the model to its own predictions.

We use a linear schedule:

$$p(e) = \begin{cases} 0 & \text{if } e < e_{\text{start}} \\ p_{\text{max}} \cdot \frac{e - e_{\text{start}}}{E - e_{\text{start}}} & \text{if } e \geq e_{\text{start}} \end{cases}$$

where $e$ is the current epoch, $e_{\text{start}}$ is `ss_start_epoch`, $E$ is the total number of epochs, and $p_{\text{max}}$ is `ss_max_prob`.

Bengio et al. (2015) explored linear, exponential, and inverse-sigmoid schedules, finding that all improved over pure teacher forcing. We use linear for simplicity and interpretability.

In [ ]:
import numpy as np

# Visualize the annealing schedule
def compute_ss_schedule(num_epochs, ss_start_epoch, ss_max_prob):
    """Compute scheduled sampling probability for each epoch."""
    ss_ramp_epochs = max(num_epochs - ss_start_epoch, 1)
    probs = []
    for epoch in range(num_epochs):
        if epoch < ss_start_epoch or ss_max_prob <= 0:
            probs.append(0.0)
        else:
            probs.append(ss_max_prob * (epoch - ss_start_epoch) / ss_ramp_epochs)
    return probs

# Example: 300 epochs, start at epoch 100, max probability 0.5
num_epochs = 300
ss_start = 100
ss_max = 0.5

schedule = compute_ss_schedule(num_epochs, ss_start, ss_max)

print(f"Schedule: {num_epochs} epochs, start at {ss_start}, max prob {ss_max}")
print(f"")
for e in [0, 50, 100, 150, 200, 250, 299]:
    print(f"  Epoch {e:3d}: ss_prob = {schedule[e]:.3f}")

## 4. Implementation Details

### 4.1 Detaching Predicted Features

When using the model's own prediction for the state update, we **detach** the predicted continuous features from the computation graph:

```python
pred_data = torch.cat([pred_cont.detach(), ss_labels.unsqueeze(-1).float()], dim=-1)
state_update = self.embed_node_feats(pred_data)
```

Without `.detach()`, gradients would flow through the prediction *and* through the state update path, creating a second gradient path through `pred_cont`. This can destabilize training because:
- The prediction is optimized by two competing objectives (minimize prediction loss *and* produce a good state update)
- The gradient magnitude approximately doubles for those parameters

By detaching, we treat the predicted values as "given inputs" — just like ground truth — for the purpose of the state update. The prediction is still trained via the normal loss; the scheduled sampling only affects what the LSTM sees as input.

### 4.2 Label Sampling

For discrete labels, we sample from the predicted distribution rather than taking the argmax:

```python
with torch.no_grad():
    ss_labels = torch.multinomial(
        F.softmax(pred_logits, dim=-1), num_samples=1
    ).squeeze(-1)
```

This mirrors the generation procedure (which also samples) and introduces beneficial stochasticity. The `torch.no_grad()` context ensures no gradients flow through the sampling operation.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# Simplified illustration of scheduled sampling in predict_node_feats
# (See bigg/extension/customized_models.py for the full implementation)

class ScheduledSamplingPredictor(nn.Module):
    def __init__(self, embed_dim, feat_dim, num_classes, ss_prob=0.0):
        super().__init__()
        self.feat_dim = feat_dim
        self.ss_prob = ss_prob

        self.feat_pred = nn.Linear(embed_dim, feat_dim)
        self.label_pred = nn.Linear(embed_dim, num_classes)
        self.feat_enc = nn.Linear(feat_dim, embed_dim)
        self.label_enc = nn.Embedding(num_classes, embed_dim)
        self.combiner = nn.Linear(embed_dim * 2, embed_dim)
        self.lstm_cell = nn.LSTMCell(embed_dim, embed_dim)

    def embed(self, feats, labels):
        e_feat = self.feat_enc(feats)
        e_label = self.label_enc(labels)
        return self.combiner(torch.cat([e_feat, e_label], dim=-1))

    def forward(self, state, target_feats, target_labels):
        h, c = state

        # Predict from hidden state
        pred_feats = self.feat_pred(h)
        pred_logits = self.label_pred(h)

        # Loss is always against ground truth
        loss_feat = F.mse_loss(pred_feats, target_feats, reduction='sum')
        loss_label = F.cross_entropy(pred_logits, target_labels, reduction='sum')

        # Scheduled sampling: choose what to feed the LSTM
        if self.training and self.ss_prob > 0 and torch.rand(1).item() < self.ss_prob:
            # Use model's own prediction (detached) for state update
            with torch.no_grad():
                sampled_labels = torch.multinomial(
                    F.softmax(pred_logits, dim=-1), num_samples=1
                ).squeeze(-1)
            state_input = self.embed(pred_feats.detach(), sampled_labels)
            used = "prediction"
        else:
            # Use ground truth for state update (teacher forcing)
            state_input = self.embed(target_feats, target_labels)
            used = "ground truth"

        new_state = self.lstm_cell(state_input, (h, c))
        return new_state, loss_feat + loss_label, used


# Demo: show that scheduled sampling stochastically swaps inputs
torch.manual_seed(0)
model = ScheduledSamplingPredictor(embed_dim=64, feat_dim=8, num_classes=4, ss_prob=0.5)
model.train()

h = torch.randn(1, 64)
c = torch.zeros(1, 64)
feats = torch.randn(1, 8)
labels = torch.tensor([2])

counts = {"ground truth": 0, "prediction": 0}
for _ in range(100):
    _, _, used = model((h, c), feats, labels)
    counts[used] += 1

print(f"Over 100 forward passes with ss_prob=0.5:")
print(f"  Used ground truth: {counts['ground truth']} times")
print(f"  Used prediction:   {counts['prediction']} times")

## 5. Combining with Hidden State Noise

Scheduled sampling and hidden state noise are complementary techniques that address the same problem from different angles:

| Technique | What it perturbs | Effect |
|---|---|---|
| Hidden state noise | The state *before* prediction | Decoders become robust to imprecise states |
| Scheduled sampling | The input *after* prediction | LSTM learns to recover from imperfect inputs |

In practice, we recommend applying both:
1. **Start with noise only** — it is simpler and less likely to destabilize training
2. **Add scheduled sampling** if noise alone is insufficient — begin annealing after the model has learned reasonable structure (e.g., `ss_start_epoch` = 30–50% of total epochs)

The noise provides a smooth, always-on regularization, while scheduled sampling introduces the more direct (but potentially destabilizing) exposure to the model's own outputs.

## 6. Usage

Both parameters are exposed as command-line arguments:

```bash
# Scheduled sampling only: ramp to 50% swap probability, starting at epoch 100
bash scripts/train/train_bigg.sh reddit 512 1 300 0.001 256 0.0 0.5 100

# Combined: noise_std=0.1 + scheduled sampling max 0.5 from epoch 100
bash scripts/train/train_bigg.sh reddit 512 1 300 0.001 256 0.1 0.5 100

# Or directly via the pipeline
python -m bigg.extension.pipeline \
  -data_dir reddit \
  -noise_std 0.1 \
  -ss_max_prob 0.5 \
  -ss_start_epoch 100 \
  ...
```

The arguments (positional order in `train_bigg.sh`):

| Position | Argument | Default | Description |
|---|---|---|---|
| 7 | `NOISE_STD` | 0.0 | Gaussian noise std on hidden state |
| 8 | `SS_MAX_PROB` | 0.0 | Maximum scheduled sampling probability |
| 9 | `SS_START_EPOCH` | 0 | Epoch to begin annealing |

## References

- Bengio, S., Vinyals, O., Jaitly, N. & Shazeer, N. (2015). *Scheduled Sampling for Sequence Prediction with Recurrent Neural Networks.* NeurIPS.
- Ranzato, M. et al. (2015). *Sequence Level Training with Recurrent Neural Networks.* ICLR.
- Lamb, A. et al. (2016). *Professor Forcing: A New Algorithm for Training Recurrent Networks.* NeurIPS.
- Dai, H. et al. (2020). *Scalable Deep Generative Modeling for Sparse Graphs.* ICML. (BiGG)